In [ ]:
# ============================================================
# Cell 0 — Notebook Registry
# Optional. Default OFF. Never appends a new row.
# Unique key: job_name + sheet_label + Tab name
# ============================================================

# OFF / CHECK / UPDATE_URL / UPDATE_URL_AND_NAME
REGISTER_MODE = "OFF"
REGISTRY_STRICT = False

REGISTRY_TAB = "Cfg__colab&sheets"
REGISTRY_JOB_NAME = "edit_metafields"
REGISTRY_SHEET_LABEL = "edit"
REGISTRY_TAB_NAME = "Edit__ValuesLong"

# Preserve the existing Colab Drive resource; only the formal notebook name changes.
CURRENT_COLAB_URL = "https://colab.research.google.com/drive/1L5sHbPw_DHv3L7XHcY0cAVVO3veTC-UH#scrollTo=pbs-edit-metafields-registry"
CURRENT_COLAB_NAME = "1.1.2 P-Edit__Metafields-Github.ipynb"

print("Notebook Registry config ready")
print("REGISTER_MODE:", REGISTER_MODE)
print(
    "logical_key:",
    REGISTRY_JOB_NAME,
    "+",
    REGISTRY_SHEET_LABEL,
    "+",
    REGISTRY_TAB_NAME,
)


In [ ]:
# ============================================================
# Cell 1 — Business Config
# Business behavior is preserved from the current Edit__ValuesLong runner.
# Account / Secret / API routing is resolved in Cell 2.
# ============================================================

# ---------- Project / Site / Job ----------
PROJECT_CODE = "PBS"
SITE_CODE = PROJECT_CODE
JOB_NAME = "edit_metafields"

# ---------- Workspace Project Registry ----------
WORKSPACE_PROJECT_REGISTRY_ID = "1-5dXTm3ZWHTVCoXB9nG7A4CtqsCZPmw-UQckWotMjwQ"
WORKSPACE_PROJECT_REGISTRY_TAB = "Cfg__Projects"
WORKSPACE_GSHEET_SECRET_NAME = "WORKSPACE_GSHEET"

# Optional Local override. Empty lets Workspace Secret Resolver use its
# configured canonical sources.
SECRET_HOME = ""

# ---------- Sheet labels / tabs ----------
INPUT_SHEET_LABEL = "edit"
WORKSHEET_TITLE = "Edit__ValuesLong"

CFG_SHEET_LABEL = "config"
CFG_TAB_FIELDS = "Cfg__Fields"

RUNLOG_SHEET_LABEL = "runlog_sheet"
RUNLOG_TAB_NAME = "Ops__RunLog"

# ---------- Type controls ----------
REFERENCE_DEFAULT_KIND = "mixed"   # "mixed" | "metaobject"
TYPE_OVERRIDE_BY_FIELD_KEY = {
    # "mf.custom.xxx": "list.metaobject_reference",
    # "v_mf.custom.yyy": "metaobject_reference",
}
ALLOW_MISSING_CFG_TYPE_FALLBACK = False

# ---------- Run controller ----------
import datetime
RUN_ID = datetime.datetime.utcnow().strftime("edit_%Y%m%d_%H%M%S")

# Safe default: first run produces Preview only. Change CONFIRMED to True
# only after value_preview shows separate JSON GIDs.
DRY_RUN = False
CONFIRMED = False
PREVIEW_LIMIT = 50

MODE_DEFAULT = "STRICT"
WRITE_MODE = "UPSERT"

ONLY_ENTITY_TYPES = None
ONLY_FIELD_PREFIXES = {"mf.", "v_mf."}

SET_BATCH_SIZE = 25
ISOLATE_SET_USER_ERRORS = True
ABORT_AFTER_FULL_FAILED_SET_BATCHES = 1
HTTP_TIMEOUT = 60
DETAIL_MAX_PER_REASON = 2

# Historical DELETE_EMPTY / ABORT_IF_FIELDKEY_CONTAINS controls were not wired
# into business behavior, so they are not exposed as operator controls here.

print("Business Config ready")
print("PROJECT_CODE:", PROJECT_CODE)
print("SITE_CODE:", SITE_CODE)
print("JOB_NAME:", JOB_NAME)
print("WORKSHEET_TITLE:", WORKSHEET_TITLE)
print("DRY_RUN:", DRY_RUN)
print("CONFIRMED:", CONFIRMED)
print("WRITE_MODE:", WRITE_MODE)
print("ALLOW_MISSING_CFG_TYPE_FALLBACK:", ALLOW_MISSING_CFG_TYPE_FALLBACK)
print("ISOLATE_SET_USER_ERRORS:", ISOLATE_SET_USER_ERRORS)
print("ABORT_AFTER_FULL_FAILED_SET_BATCHES:", ABORT_AFTER_FULL_FAILED_SET_BATCHES)
print("ONLY_ENTITY_TYPES:", ONLY_ENTITY_TYPES)
print("ONLY_FIELD_PREFIXES:", ONLY_FIELD_PREFIXES)


In [ ]:
# ============================================================
# Cell 2 — Runtime / Dependency / Auth / Module / Registry
# Colab: clean clone from GitHub. Local: formal Console_Core only.
# ============================================================

import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nikkilog/ecom.git"
REPO_BRANCH = "main"
LOCAL_REPO_DIR = Path("/Users/nikki/Documents/AI_Workspace/Projects/Console_Core")
MODULE_PATH = "shopify_sync.1_1_1_edit_metafields"
COLAB_AUTO_INSTALL_MISSING_DEPENDENCIES = True


def _module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


IN_COLAB = _module_available("google.colab")
RUNTIME_MODE = "COLAB" if IN_COLAB else "LOCAL"

DEPENDENCIES = {
    "gspread": "gspread",
    "google.auth": "google-auth",
    "pandas": "pandas",
    "requests": "requests",
}
missing_packages = sorted({
    package
    for module_name, package in DEPENDENCIES.items()
    if not _module_available(module_name)
})

if missing_packages:
    command = [sys.executable, "-m", "pip", "install", *missing_packages]
    if IN_COLAB and COLAB_AUTO_INSTALL_MISSING_DEPENDENCIES:
        print("[1/7] Runtime | installing missing Colab dependencies:", missing_packages)
        subprocess.run(command, check=True)
    else:
        raise RuntimeError(
            "Missing Local dependencies: "
            + ", ".join(missing_packages)
            + "\nInstall once with:\n"
            + " ".join(command)
        )
else:
    print("[1/7] Runtime | dependency preflight: PASS")

if not IN_COLAB and not _module_available("workspace_secret_resolver"):
    resolver_dir = Path(
        os.environ.get(
            "WORKSPACE_SECRET_RESOLVER_DIR",
            "/Users/nikki/Documents/AI_Workspace/Projects/Workspace_Secret_Resolver",
        )
    ).expanduser()
    raise RuntimeError(
        "Missing Local dependency: workspace-secret-resolver.\n"
        "Install once into this exact kernel with:\n"
        f"{sys.executable} -m pip install -e {resolver_dir}"
    )
elif not IN_COLAB:
    print("[2/7] Runtime | Workspace Secret Resolver: PASS")
else:
    print("[2/7] Runtime | Colab native Secret adapter selected")


def _run_cmd(args, cwd=None):
    print(">", " ".join(str(value) for value in args))
    subprocess.run([str(value) for value in args], cwd=cwd, check=True)


if IN_COLAB:
    REPO_DIR = Path("/content/ecom")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print("[3/7] Repository | clean Colab clone from GitHub")
    _run_cmd([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
        REPO_URL, REPO_DIR,
    ])
else:
    REPO_DIR = LOCAL_REPO_DIR.expanduser().resolve()
    expected_module_file = REPO_DIR / "shopify_sync" / "1_1_1_edit_metafields.py"
    if not REPO_DIR.is_dir():
        raise RuntimeError(
            "Formal Local Console_Core repository does not exist:\n"
            f"{REPO_DIR}"
        )
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            "Formal Local Console_Core path is not a Git checkout:\n"
            f"{REPO_DIR}"
        )
    if not expected_module_file.is_file():
        raise RuntimeError(
            "Current Edit Metafields module is missing from the formal Local repository:\n"
            f"{expected_module_file}"
        )
    print("[3/7] Repository | using formal Local Console_Core checkout:", REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Numeric-leading filename: import through the full module string.
# Clear historical module identities to prevent stale-kernel reuse.
importlib.invalidate_caches()
for loaded_name in [
    MODULE_PATH,
    "shopify_sync.edit_metafields",
    "shopify_sync.sync_metafields",
    "shopify_sync",
]:
    if loaded_name in sys.modules:
        del sys.modules[loaded_name]

edit_metafields = importlib.import_module(MODULE_PATH)
edit_metafields = importlib.reload(edit_metafields)

commit_sha = "UNKNOWN"
try:
    commit_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO_DIR,
        text=True,
    ).strip()
except Exception:
    pass

print("[4/7] Module | loaded Current")
print("RUNTIME_MODE:", RUNTIME_MODE)
print("REPOSITORY_SOURCE:", "GITHUB_CLONE" if IN_COLAB else "LOCAL_CONSOLE_CORE")
print("REPO_DIR:", REPO_DIR)
print("GIT_COMMIT:", commit_sha)
print("MODULE_PATH:", getattr(edit_metafields, "MODULE_PATH", MODULE_PATH))
print("MODULE_FILE:", edit_metafields.__file__)
print("MODULE_VERSION:", getattr(edit_metafields, "MODULE_VERSION", "UNKNOWN"))

# MODULE_VERSION is provenance only. It is printed above but is not a runtime gate.
# The stable module identity is MODULE_PATH + MODULE_FILE.

print("[5/7] Project/Auth | resolving Workspace Registry and project credentials")
RUNTIME_CONTEXT = edit_metafields.resolve_runtime_context(
    project_code=PROJECT_CODE,
    workspace_registry_id=WORKSPACE_PROJECT_REGISTRY_ID,
    workspace_gsheet_secret_name=WORKSPACE_GSHEET_SECRET_NAME,
    workspace_registry_tab=WORKSPACE_PROJECT_REGISTRY_TAB,
    secret_home=SECRET_HOME or None,
    print_progress=True,
)

PROJECT_ROUTE = RUNTIME_CONTEXT["project_route"]
ACCOUNT_CONFIG = RUNTIME_CONTEXT["account"]
AUTH_META = RUNTIME_CONTEXT["auth"]

CONSOLE_CORE_URL = PROJECT_ROUTE["console_core_url"]
GSHEET_SA_SECRET_NAME = ACCOUNT_CONFIG["gsheet_secret_name"]
SHOPIFY_TOKEN_SECRET_NAME = ACCOUNT_CONFIG["shopify_token_secret_name"]
SHOP_DOMAIN = ACCOUNT_CONFIG["shop_domain"]
API_VERSION = ACCOUNT_CONFIG["api_version"]
TAB_CFG_ACCOUNT_ID = PROJECT_ROUTE["account_config_tab"]
TZ_NAME = PROJECT_ROUTE["timezone"]

# Resolved Secret values are intentionally not printed.
GSHEET_CREDENTIAL_VALUE = RUNTIME_CONTEXT["credentials"]["gsheet_sa_value"]
SHOPIFY_ACCESS_TOKEN = RUNTIME_CONTEXT["credentials"]["shopify_access_token"]

print("[Project route]")
print("  PROJECT_CODE:", PROJECT_ROUTE["project_code"])
print("  PROJECT_NAME:", PROJECT_ROUTE["project_name"])
print("  CONSOLE_CORE_URL:", CONSOLE_CORE_URL)
print("  GSHEET_SA_SECRET_NAME:", GSHEET_SA_SECRET_NAME)
print("  SHOPIFY_TOKEN_SECRET_NAME:", SHOPIFY_TOKEN_SECRET_NAME)
print("  TAB_CFG_ACCOUNT_ID:", TAB_CFG_ACCOUNT_ID)
print("  TZ_NAME:", TZ_NAME)
print("  SHOP_DOMAIN:", SHOP_DOMAIN)
print("  SHOPIFY_API_VERSION:", API_VERSION)
print("  GOOGLE_SECRET_SOURCE:", AUTH_META["project_google_secret_source_type"])
print("  SHOPIFY_SECRET_SOURCE:", AUTH_META["shopify_secret_source_type"])

print("[6/7] Registry | checking configured existing row")
try:
    REGISTRY_RESULT = edit_metafields.update_existing_notebook_registry_row(
        project_code=PROJECT_CODE,
        registry_mode=REGISTER_MODE,
        console_core_url=CONSOLE_CORE_URL,
        bootstrap_gsheet_secret_name=GSHEET_SA_SECRET_NAME,
        registry_tab=REGISTRY_TAB,
        job_name=REGISTRY_JOB_NAME,
        sheet_label=REGISTRY_SHEET_LABEL,
        tab_name=REGISTRY_TAB_NAME,
        current_colab_url=CURRENT_COLAB_URL,
        current_colab_name=CURRENT_COLAB_NAME,
        secret_home=SECRET_HOME or None,
        print_progress=True,
    )
except Exception as exc:
    if REGISTRY_STRICT:
        raise
    REGISTRY_RESULT = {
        "status": "WARNING",
        "changed_fields": [],
        "target_row": None,
        "warning": f"{type(exc).__name__}: {exc}",
    }
    print("[Registry warning]", REGISTRY_RESULT["warning"])

print("[Registry result]")
print(REGISTRY_RESULT)
print("[7/7] Runtime/Auth ready")


In [ ]:
# ============================================================
# Cell 3 — Execution / Progress / Result
# Business algorithm lives only in the formal Python module.
# ============================================================

from IPython.display import display
import pandas as pd
import time

print("=" * 72)
print("START | edit_metafields")
print("=" * 72)
print("RUNTIME_MODE:", RUNTIME_MODE)
print("PROJECT_CODE:", PROJECT_CODE)
print("SITE_CODE:", SITE_CODE)
print("MODULE_PATH:", MODULE_PATH)
print("MODULE_FILE:", edit_metafields.__file__)
print("MODULE_VERSION:", getattr(edit_metafields, "MODULE_VERSION", "UNKNOWN"))
print("GIT_COMMIT:", commit_sha)
print("SHOP_DOMAIN:", SHOP_DOMAIN)
print("SHOPIFY_API_VERSION:", API_VERSION)
print("INPUT_OBJECT:", f"{INPUT_SHEET_LABEL} / {WORKSHEET_TITLE}")
print("CFG_OBJECT:", f"{CFG_SHEET_LABEL} / {CFG_TAB_FIELDS}")
print("RUNLOG_OBJECT:", f"{RUNLOG_SHEET_LABEL} / {RUNLOG_TAB_NAME}")
print("DRY_RUN:", DRY_RUN)
print("CONFIRMED:", CONFIRMED)
print("WRITE_MODE:", WRITE_MODE)
print()

started = time.time()

try:
    result = edit_metafields.run(
        site_code=SITE_CODE,
        job_name=JOB_NAME,

        gsheet_sa_value=GSHEET_CREDENTIAL_VALUE,
        shopify_access_token=SHOPIFY_ACCESS_TOKEN,
        shop_domain=SHOP_DOMAIN,
        api_version=API_VERSION,

        console_core_url=CONSOLE_CORE_URL,
        input_sheet_label=INPUT_SHEET_LABEL,
        worksheet_title=WORKSHEET_TITLE,

        cfg_sheet_label=CFG_SHEET_LABEL,
        cfg_tab_fields=CFG_TAB_FIELDS,

        runlog_sheet_label=RUNLOG_SHEET_LABEL,
        runlog_tab_name=RUNLOG_TAB_NAME,

        run_id=RUN_ID,
        dry_run=DRY_RUN,
        confirmed=CONFIRMED,
        preview_limit=PREVIEW_LIMIT,

        mode_default=MODE_DEFAULT,
        write_mode=WRITE_MODE,
        only_entity_types=ONLY_ENTITY_TYPES,
        only_field_prefixes=ONLY_FIELD_PREFIXES,

        reference_default_kind=REFERENCE_DEFAULT_KIND,
        type_override_by_field_key=TYPE_OVERRIDE_BY_FIELD_KEY,
        allow_missing_cfg_type_fallback=ALLOW_MISSING_CFG_TYPE_FALLBACK,

        set_batch_size=SET_BATCH_SIZE,
        isolate_set_user_errors=ISOLATE_SET_USER_ERRORS,
        abort_after_full_failed_set_batches=ABORT_AFTER_FULL_FAILED_SET_BATCHES,
        http_timeout=HTTP_TIMEOUT,
        detail_max_per_reason=DETAIL_MAX_PER_REASON,
    )
except Exception as exc:
    print()
    print("=" * 72)
    print("FAILED")
    print("=" * 72)
    print(type(exc).__name__ + ":", exc)
    raise

elapsed = time.time() - started

status = result.get("status")
meta = result.get("meta", {}) or {}
summary = result.get("summary", {}) or {}
preview = result.get("preview", []) or []
warnings = result.get("warnings", []) or []

print()
print("=" * 72)
print("FINAL RESULT")
print("=" * 72)
print("status:", status)
print("elapsed_seconds:", round(elapsed, 1))

print("\nMeta")
if meta:
    for key, value in meta.items():
        print(f"- {key}: {value}")
else:
    print("(none)")

print("\nSummary")
if summary:
    for key, value in summary.items():
        print(f"- {key}: {value}")
else:
    print("(none)")

print("\nPreview")
if preview:
    preview_df = pd.DataFrame(preview)
    print("Preview rows:", len(preview_df))
    print("Preview columns:", list(preview_df.columns))
    display(preview_df)
else:
    print("(none)")

print("\nWarnings")
if warnings:
    for item in warnings:
        print("-", item.get("type", "(unknown)"), ":", item.get("count", 0))
        examples = item.get("examples") or []
        if examples:
            display(pd.DataFrame(examples[:2]))
else:
    print("(none)")

print("DONE")
